In [7]:
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Colab_Notebooks/Milking'
print('Project directory set to:', PROJECT_DIR)
%cd "$PROJECT_DIR"
!ls -la


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory set to: /content/drive/MyDrive/Colab_Notebooks/Milking
/content/drive/MyDrive/Colab_Notebooks/Milking
total 6434
-rw------- 1 root root     182 Nov 27 10:08 data.yaml
-rw------- 1 root root   18975 Nov 27 10:12 Milking_training.ipynb
-rw------- 1 root root     165 Nov 26 11:08 README.dataset.txt
-rw------- 1 root root    1002 Nov 26 11:08 README.roboflow.txt
drwx------ 3 root root    4096 Nov 27 10:08 runs
drwx------ 3 root root    4096 Nov 27 09:23 test
drwx------ 3 root root    4096 Nov 26 11:18 train
drwx------ 3 root root    4096 Nov 27 09:23 valid
-rw------- 1 root root 6549796 Oct  3 05:26 yolov8n.pt


In [8]:
# Install ultralytics (YOLOv8). Pin a stable version known to work well on Colab.
!pip install -q ultralytics==8.3.30

import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
!nvidia-smi -L || true


Torch version: 2.9.0+cu126
CUDA available: True
GPU 0: Tesla T4 (UUID: GPU-09f391fa-7ea3-4b1f-1932-344448d3d664)


In [10]:
# Prepare dataset: fix data.yaml relative paths and create valid/test from train if missing.
import yaml, shutil, os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Colab_Notebooks/Milking')
DATA_YAML_PATH = PROJECT_DIR / 'data.yaml'
print('Checking', DATA_YAML_PATH)

if not DATA_YAML_PATH.exists():
    raise FileNotFoundError(f'{DATA_YAML_PATH} not found. Please upload your data.yaml into the folder.')

with open(DATA_YAML_PATH, 'r') as f:
    data_cfg = yaml.safe_load(f)

for key in ['train', 'val', 'test']:
    if key in data_cfg and data_cfg[key]:
        p = data_cfg[key]
        if '..' in str(p):
            segs = Path(p).parts
            data_cfg[key] = str(Path(*segs[-2:]))
        else:
            data_cfg[key] = str(Path(p))

with open(DATA_YAML_PATH, 'w') as f:
    yaml.safe_dump(data_cfg, f)

print('Wrote fixed data.yaml:')
print(open(DATA_YAML_PATH).read())

train_images = PROJECT_DIR / 'train' / 'images'
train_labels = PROJECT_DIR / 'train' / 'labels'
for split in ['valid', 'test']:
    split_images = PROJECT_DIR / split / 'images'
    split_labels = PROJECT_DIR / split / 'labels'
    if not split_images.exists():
        print(f'Creating {split}/ from train/ images and labels...')
        shutil.copytree(train_images, split_images, dirs_exist_ok=True)
        if train_labels.exists():
            shutil.copytree(train_labels, split_labels, dirs_exist_ok=True)
        else:
            print('Warning: train/labels does not exist — make sure labels are present for training.')

print('Dataset folders:')
for p in ['train', 'valid', 'test']:
    print(p, 'images:', len(list((PROJECT_DIR / p / 'images').glob('*'))))


Checking /content/drive/MyDrive/Colab_Notebooks/Milking/data.yaml
Wrote fixed data.yaml:
names:
- cluster_attached
- cluster_detached
- cow_leg_udder
path: /content/drive/MyDrive/Colab_Notebooks/Milking/Milking_activity 2.v1i.yolov8
train: train/images
val: valid/images

Dataset folders:
train images: 497
valid images: 497
test images: 497


In [13]:
# Training with ultralytics YOLOv8
from ultralytics import YOLO
import os
import yaml # Added import for yaml

PROJECT_DIR = '/content/drive/MyDrive/Colab_Notebooks/Milking'
DATA_YAML = os.path.join(PROJECT_DIR, 'data.yaml')

# Model choices: 'yolov8n.pt', 'yolov8s.pt', etc.
MODEL = os.path.join(PROJECT_DIR, 'yolov8n.pt')  # you already have this; otherwise use 'yolov8n.pt' to download
EPOCHS = 50
BATCH = 8
IMGSZ = 312
PROJECT = os.path.join(PROJECT_DIR, 'runs')
NAME = 'scraping_yolov8_train'

print('Model:', MODEL)
print('Data yaml:', DATA_YAML)
print('Project dir:', PROJECT)

model = YOLO(MODEL)

# Load data.yaml content and correct the 'path' entry
with open(DATA_YAML, 'r') as f:
    data_cfg_corrected = yaml.safe_load(f)

# Set the base path in data.yaml to the actual PROJECT_DIR
data_cfg_corrected['path'] = str(PROJECT_DIR)

# Save the corrected data_cfg back to data.yaml before training
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data_cfg_corrected, f)

results = model.train(
    data=DATA_YAML, # Pass the path string to the corrected data.yaml file
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device='cuda',
    name=NAME,
    project=PROJECT,
    patience=10,
    optimizer='SGD',
    cache=True,
    workers=2,
    close_mosaic=10
)

print('Training finished. Check runs folder in Drive for outputs.')

Model: /content/drive/MyDrive/Colab_Notebooks/Milking/yolov8n.pt
Data yaml: /content/drive/MyDrive/Colab_Notebooks/Milking/data.yaml
Project dir: /content/drive/MyDrive/Colab_Notebooks/Milking/runs
New https://pypi.org/project/ultralytics/8.3.232 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.30 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=/content/drive/MyDrive/Colab_Notebooks/Milking/yolov8n.pt, data=/content/drive/MyDrive/Colab_Notebooks/Milking/data.yaml, epochs=50, time=None, patience=10, batch=8, imgsz=312, save=True, save_period=-1, cache=True, device=cuda, workers=2, project=/content/drive/MyDrive/Colab_Notebooks/Milking/runs, name=scraping_yolov8_train3, exist_ok=False, pretrained=True, optimizer=SGD, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overl

100%|██████████| 755k/755k [00:00<00:00, 26.4MB/s]


Overriding model.yaml nc=80 with nc=3

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

100%|██████████| 5.35M/5.35M [00:00<00:00, 93.7MB/s]


AMP: checks passed ✅
WARNING ⚠️ imgsz=[312] must be multiple of max stride 32, updating to [320]


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      50/50     0.346G      1.187     0.6942     0.9442          2        320: 100%|██████████| 63/63 [00:04<00:00, 12.62it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 32/32 [00:02<00:00, 10.70it/s]


                   all        497       2918      0.898      0.745      0.862      0.552

50 epochs completed in 0.133 hours.
Optimizer stripped from /content/drive/MyDrive/Colab_Notebooks/Milking/runs/scraping_yolov8_train3/weights/last.pt, 6.2MB
Optimizer stripped from /content/drive/MyDrive/Colab_Notebooks/Milking/runs/scraping_yolov8_train3/weights/best.pt, 6.2MB

Validating /content/drive/MyDrive/Colab_Notebooks/Milking/runs/scraping_yolov8_train3/weights/best.pt...
Ultralytics 8.3.30 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 168 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 32/32 [00:04<00:00,  7.07it/s]


                   all        497       2918      0.897      0.747      0.861      0.551
      cluster_attached        344        907      0.988      0.647      0.843      0.502
      cluster_detached         57        163      0.728      0.674      0.757      0.468
         cow_leg_udder        448       1848      0.974      0.919      0.984      0.684
Speed: 0.1ms preprocess, 1.1ms inference, 0.0ms loss, 2.4ms postprocess per image
Results saved to /content/drive/MyDrive/Colab_Notebooks/Milking/runs/scraping_yolov8_train3
Training finished. Check runs folder in Drive for outputs.
